# v0.6.0 vs v0.5.0 Validation Report

This notebook documents the v0.6.0 milestone validation. It compares v0.6.0 against
the v0.5.0 baseline on three axes:

1. **Fit-quality parity** — per-peak column values on the `small_3x4` synthetic cube
2. **Runtime comparison** — `n_curve_fit_calls` and wall-clock time vs. v0.5.0 baseline
3. **Failure-mode comparison** — failed-pixel counts on a hard cube with vs. without peak proposals

All numbers are generated by `benchmarks/validation_v0.6.0_vs_v0.5.0.py` using deterministic
synthetic cubes (`random_state=42`, `numpy.random.default_rng(42)`). Run that script first
to regenerate the CSV and JSON inputs.

**Prerequisites:** `pip install -e .` (base install — no `[ml]` extra required).

## 1. Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("../..")
RESULTS_DIR = REPO_ROOT / "benchmarks" / "results"
BASELINE_DIR = RESULTS_DIR / "v0.5.0_baseline"

df = pd.read_csv(RESULTS_DIR / "v0.6.0_validation.csv")
with open(RESULTS_DIR / "v0.6.0_validation_summary.json") as f:
    summary = json.load(f)
with open(BASELINE_DIR / "v0.5.0_baseline.json") as f:
    baseline = json.load(f)
checksums_sha256 = (BASELINE_DIR / "checksums.txt").read_text().strip().split(":")[1].split()[0]

print(f"Loaded {len(df)} validation rows")
print(f"Datasets: {df['dataset_name'].unique().tolist()}")
df.head()

## 2. Fit-quality parity

The byte-level export parity check on the `small_3x4` cube is the primary scientific-parity
invariant: it certifies that the fitted peak parameters are numerically identical (within
floating-point tolerance across LAPACK builds, `rtol=1e-3, atol=1e-3`) to the v0.5.0 reference.

This is the same criterion enforced automatically by
`tests/test_export_fit_map_qa_columns.py::test_wide_format_existing_columns_unchanged`.

In [ ]:
fq = summary["fit_quality"]
parity_status = "PASS" if fq["parity"] else "FAIL"

print("=== Fit-quality parity ===")
print(f"  v0.5.0 SHA256 (reference)  : {fq['export_sha256_v0_5_0']}")
print(f"  v0.6.0 SHA256 (current)    : {fq['export_sha256_current']}")
print(f"  Per-peak column parity     : {parity_status}")
print()
print("Note: SHA256 values differ because v0.6.0 export includes QA columns (rmse, ok,")
print("n_starts, n_params_at_bounds) added in v0.5.1. Parity is assessed on per-peak")
print("columns only, matching the v0.5.0 schema.")

In [ ]:
# Per-cube mean_rmse_finite table
rmse_table = (
    df[df["use_peak_proposals"] == True]
    .groupby("dataset_name")[["mean_rmse_finite", "success_rate", "n_failed_pixels"]]
    .mean()
    .round(6)
)
print("Per-cube average fit quality (standard grid, use_peak_proposals=True):")
rmse_table

## 3. Runtime comparison

**Advisory note (from `benchmark_mapping_fit.py`):** Wall-clock `runtime_s` is advisory
because scipy iteration count, warm-start state, and adaptive multistart all introduce
structural variability that is not noise but algorithmic state. A 1.0x–1.2x spread
between runs is expected. `n_curve_fit_calls` is the hardware-independent primary metric;
algorithmic improvements should show a reduction here.

In [ ]:
rt = summary["runtime"]
print("=== Runtime comparison (small_3x4, warm_start=False, n_starts=1) ===")
print(f"  v0.5.0 baseline median runtime : {rt['v0_5_0_baseline_median_s']:.4f} s")
print(f"  v0.5.0 n_curve_fit_calls       : {baseline['n_curve_fit_calls']}")
print(f"  v0.6.0 runtime                 : {rt['current_median_s']:.4f} s")
baseline_row = df[(df["dataset_name"] == "small_3x4") & (df["warm_start"] == False) & (df["n_starts"] == 1)].iloc[0]
v06_calls = int(baseline_row["n_curve_fit_calls"])
print(f"  v0.6.0 n_curve_fit_calls       : {v06_calls}")
print(f"  runtime ratio (v0.6.0/v0.5.0)  : {rt['ratio']:.3f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

versions = ["v0.5.0", "v0.6.0"]

# n_curve_fit_calls
calls = [baseline["n_curve_fit_calls"], v06_calls]
axes[0].bar(versions, calls, color=["steelblue", "coral"])
axes[0].set_ylabel("n_curve_fit_calls")
axes[0].set_title("Curve-fit calls (small_3x4, n_starts=1)")
for i, v in enumerate(calls):
    axes[0].text(i, v + 0.2, str(v), ha="center", va="bottom", fontsize=11)

# Wall-clock runtime
runtimes = [rt["v0_5_0_baseline_median_s"], rt["current_median_s"]]
axes[1].bar(versions, runtimes, color=["steelblue", "coral"])
axes[1].set_ylabel("Runtime (s)")
axes[1].set_title("Wall-clock runtime (advisory)")
for i, v in enumerate(runtimes):
    axes[1].text(i, v + 0.005, f"{v:.3f}s", ha="center", va="bottom", fontsize=11)

fig.suptitle("Runtime: v0.6.0 vs v0.5.0 — small_3x4 cube", fontsize=12)
plt.tight_layout()
plt.show()
print("Note: n_curve_fit_calls is the hardware-independent primary metric.")

## 4. Failure-mode comparison

A hard synthetic cube (two overlapping peaks at 500 and 530 cm⁻¹, SNR≈1.7, `maxfev=50`)
is used to measure the effect of the peak-proposal fallback (v0.5.3). This reproduces
and formalises the hard-cube evidence cited in the v0.5.3 CHANGELOG as a citable artefact.

With `use_peak_proposals=True` (the v0.5.3+ default), the proposal fallback is invoked only
after all existing adaptive-multistart and warm-start paths have failed.

In [ ]:
fm = summary["failure_mode"]
failed_no_pp = fm["hard_cube_failed_pixels_without_proposals"]
failed_with_pp = fm["hard_cube_failed_pixels_with_proposals"]

print("=== Failure-mode comparison (hard_3x4 cube, maxfev=50) ===")
print(f"  Failed pixels (use_peak_proposals=False) : {failed_no_pp}")
print(f"  Failed pixels (use_peak_proposals=True)  : {failed_with_pp}")
delta = failed_no_pp - failed_with_pp
print(f"  Reduction                                : {delta} pixels recovered")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
labels = ["Without proposals\n(use_peak_proposals=False)", "With proposals\n(use_peak_proposals=True)"]
values = [failed_no_pp, failed_with_pp]
bars = ax.bar(labels, values, color=["tomato", "mediumseagreen"])
ax.set_ylabel("Failed pixels")
ax.set_title("Failure-mode: hard_3x4 cube (overlapping peaks, maxfev=50)")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.05, str(v), ha="center", va="bottom", fontsize=12)
plt.tight_layout()
plt.show()

hard_df = df[df["dataset_name"] == "hard_3x4"][["use_peak_proposals", "n_failed_pixels", "success_rate", "n_curve_fit_calls"]]
print("\nHard cube detail:")
hard_df